In [3]:
import pandas as pd
import numpy as np

codes= get_index_stocks('000300.XSHG',date="2025-1-1")
codes_1=codes[:10]

start_date = '2025-01-01'
end_date = '2026-01-1'

stocks=codes_1
df = get_price(
    security=stocks,    # 传入股票列表
    start_date=start_date,
    end_date=end_date,
    frequency='daily',    # daily=日线, minute=分钟线
    fields=['open','close', 'volume'],# 需要的字段
    panel=False
)
df['time'] = pd.to_datetime(df['time'])

print(df.head())
print(df.tail())
print(df.info())

df["日内收益率"] = (df["close"] - df["open"]) / df["open"]

print("计算完成，收益率结果：")
print(df.round(4).head())
print(df.round(4).tail())
print(df.round(4).info())

query_object=query(valuation.circulating_market_cap,
                   valuation.turnover_ratio,
                   valuation.pe_ratio,
                   valuation.pb_ratio,
                   valuation.ps_ratio,
                   valuation.pcf_ratio,
                   indicator.roe,
                   indicator.net_profit_margin,
                   indicator.inc_net_profit_year_on_year,
                   cash_flow.net_operate_cash_flow
                  ).filter(valuation.code.in_(stocks))
df1=get_fundamentals_continuously(query_object, end_date=end_date,count=250, panel=False)
df1 = df1.rename(columns={"day": "time"})
df1['time'] = pd.to_datetime(df1['time'])

print(df1.round(4).head())
print(df1.round(4).tail())
print(df1.round(4).info())

df_merge = pd.merge(
    left=df,        # 第一个表
    right=df1,       # 第二个表
    on=['time','code'],       # 【固定合并列】按date列匹配
    how='inner'      # 连接方式（默认inner，可修改）
)

print("按date列合并结果：")
print(df_merge.round(4).head())
print(df_merge.round(4).tail())
print(df_merge.round(4).info())

# 1. 初始化【DataFrame】，而不是字典！！！
df2 = pd.DataFrame({
    "code": stocks  # 先只放股票代码，行业列后续填充
})
# 新增空的行业列
df2["一级行业"] = ""
industry_dict=get_industry(security=stocks, date=end_date)
for code in stocks:
    industry_namex=industry_dict[code]['sw_l1']['industry_name']
    df2.loc[df2["code"] == code, "一级行业"]=industry_namex
print(df2)

df_merge2 = pd.merge(
    left=df_merge,        # 第一个表
    right=df2,       # 第二个表
    on=['code'],       # 【固定合并列】按date列匹配
    how='inner'      # 连接方式（默认inner，可修改）
)

print("按date列合并结果：")
print(df_merge2.head())
print(df_merge2.tail())
print(df_merge2.info())

        time         code   open  close       volume
0 2025-01-02  000001.XSHE  11.14  10.85  191635948.0
1 2025-01-03  000001.XSHE  10.86  10.81  121608401.0
2 2025-01-06  000001.XSHE  10.81  10.86  114326293.0
3 2025-01-07  000001.XSHE  10.84  10.93   78763272.0
4 2025-01-08  000001.XSHE  10.92  10.92  111888155.0
           time         code   open  close      volume
2425 2025-12-25  000408.XSHE  77.69  78.25  12514509.0
2426 2025-12-26  000408.XSHE  79.65  83.87  17982005.0
2427 2025-12-29  000408.XSHE  86.01  81.16  18807520.0
2428 2025-12-30  000408.XSHE  79.98  82.46  13948513.0
2429 2025-12-31  000408.XSHE  82.37  82.96  13658423.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2430 entries, 0 to 2429
Data columns (total 5 columns):
time      2430 non-null datetime64[ns]
code      2430 non-null object
open      2430 non-null float64
close     2430 non-null float64
volume    2430 non-null float64
dtypes: datetime64[ns](1), float64(3), object(1)
memory usage: 95.0+ KB
None
计算完

In [4]:
#共同拥有的列
key_cols = ["time", "code"]

#目标列
goal_cols = ["日内收益率"]

#提取目标表格
df_selected = df_merge2[goal_cols+key_cols]

#提取剩余列表格
df_remaining = df_merge2.drop(goal_cols, axis=1)

print(df_selected.head())
print(df_remaining.head())
print(df_remaining.info())

      日内收益率       time         code
0 -0.026032 2025-01-02  000001.XSHE
1 -0.004604 2025-01-03  000001.XSHE
2  0.004625 2025-01-06  000001.XSHE
3  0.008303 2025-01-07  000001.XSHE
4  0.000000 2025-01-08  000001.XSHE
        time         code  ...   net_operate_cash_flow  一级行业
0 2025-01-02  000001.XSHE  ...            2.343600e+10   银行I
1 2025-01-03  000001.XSHE  ...            2.343600e+10   银行I
2 2025-01-06  000001.XSHE  ...            2.343600e+10   银行I
3 2025-01-07  000001.XSHE  ...            2.343600e+10   银行I
4 2025-01-08  000001.XSHE  ...            2.343600e+10   银行I

[5 rows x 16 columns]
<class 'pandas.core.frame.DataFrame'>
Int64Index: 2430 entries, 0 to 2429
Data columns (total 16 columns):
time                           2430 non-null datetime64[ns]
code                           2430 non-null object
open                           2430 non-null float64
close                          2430 non-null float64
volume                         2430 non-null float64
circulating_marke

In [5]:
# MAD:中位数去极值
def extreme_MAD(dt,n = 5.2):
    median = dt.quantile(0.5)  # 找出中位数
    new_median = (abs((dt - median)).quantile(0.5))  # 偏差值的中位数
    dt_up = median + n*new_median    # 上限
    dt_down = median - n*new_median  # 下限
    return dt.clip(dt_down, dt_up, axis=1)   # 超出上下限的值，赋值为上下限

df_remaining.iloc[:,2:14] = extreme_MAD(df_remaining.iloc[:,2:14],n = 5.)
df_remaining = df_remaining.ffill()
df_remaining = df_remaining.fillna(0)

df3 = df_remaining.merge(df_selected, on=["time","code"])
df3 = df3.dropna()
#df3 = df3.sort_values(by=["time", "code"])
print(df3.head())

        time         code   open    ...     net_operate_cash_flow  一级行业     日内收益率
0 2025-01-02  000001.XSHE  11.14    ...              2.343600e+10   银行I -0.026032
1 2025-01-03  000001.XSHE  10.86    ...              2.343600e+10   银行I -0.004604
2 2025-01-06  000001.XSHE  10.81    ...              2.343600e+10   银行I  0.004625
3 2025-01-07  000001.XSHE  10.84    ...              2.343600e+10   银行I  0.008303
4 2025-01-08  000001.XSHE  10.92    ...              2.343600e+10   银行I  0.000000

[5 rows x 17 columns]


In [13]:
import statsmodels.api as sm
import numpy as np

def OlsResid(y, x):
    df = pd.concat([y, x], axis = 1)
    if df.dropna().shape[0]>0:
        resid = sm.OLS(y, x, missing='drop').fit().resid
        return resid.reindex(df.index)
    else:
        return y
    
def winsor(x):
    if x.dropna().shape[0] != 0:
        x.loc[x < np.percentile(x.dropna(),5)] = np.percentile(x.dropna(),5)
        x.loc[x > np.percentile(x.dropna(),95)] = np.percentile(x.dropna(),95)
    else:
        x = x.fillna(0)
    return x
    
def norm(data, if_neutral):
    data = data.copy()
    """
    数据预处理，标准化
    """
    # 判断有无缺失值，若有缺失值，drop，若都缺失，返回原值
    datax = data.copy()
    if data.shape[0] != 0:
        classname = data['一级行业']
        mkt = data['circulating_market_cap']
        data = data.drop(['一级行业','circulating_market_cap'],axis = 1)
        ## 去极值
        data = data.apply(lambda x:winsor(x),axis = 0)
        ## 中性化
        if if_neutral: # 是否中性
            class_var = pd.get_dummies(classname,columns=['一级行业'],prefix='一级行业', prefix_sep="_", dummy_na=False, drop_first=True)
            # 原来是log这里市值取对数出现了问题！！！！log(0)=-inf
            class_var['circulating_market_cap'] = np.log(mkt).replace(float('-inf'),0)
            class_var['Intercept'] = 1
            x = class_var
            # 每个因子对所有自变量做回归，得到残差值
            data = data.apply(func = OlsResid, args = (x,), axis = 0)

        ## zscore
        data1 = (data - data.mean())/data.std()

        # 缺失部分补进去
        data1 = data1.reindex(datax.index)
    else:
        data1 = data
    return data1

# Z值标准化
def standardize_z(dt):
    mean = dt.mean()        # 截面数据均值
    std = dt.std()          # 截面数据标准差
    return (dt - mean)/std

def NormFactors(datas, if_neutral):
    datas_copy = datas.copy()
    datas_copy_0=datas_copy[["time","code","circulating_market_cap"]]
    datas_copy_0.iloc[:, 2] = standardize_z(datas_copy_0.iloc[:, 2])
    fnormall = []
    dates = datas["time"].unique()
    for dateuse in dates: # dateuse = dates[0]
        datause = datas.loc[datas["time"] == dateuse]
        stockname = datause[['time','code','close','日内收益率']]
        fnorm = norm(datause.drop(['time','code','close','日内收益率'],axis = 1),if_neutral)
        fnormall.append(pd.concat([stockname, fnorm],axis = 1))
        # print(fnormall)
        print('{}中性化完成！'.format(dateuse))
    fnormall = pd.concat(fnormall,axis = 0)
    fnormall = pd.merge(
    left=fnormall, 
    right=datas_copy_0, 
    on=['time', 'code'],  # 关键：传入两个共同列名的列表
    how='inner'  # 合并方式（默认inner，可修改）
    )
    fnormall = fnormall.sort_values(by = ['time','code'])
    return fnormall.reset_index(drop = True)



In [15]:
df4=NormFactors(df3,if_neutral=True)

2025-01-02T00:00:00.000000000中性化完成！
2025-01-03T00:00:00.000000000中性化完成！
2025-01-06T00:00:00.000000000中性化完成！
2025-01-07T00:00:00.000000000中性化完成！
2025-01-08T00:00:00.000000000中性化完成！
2025-01-09T00:00:00.000000000中性化完成！
2025-01-10T00:00:00.000000000中性化完成！
2025-01-13T00:00:00.000000000中性化完成！
2025-01-14T00:00:00.000000000中性化完成！
2025-01-15T00:00:00.000000000中性化完成！
2025-01-16T00:00:00.000000000中性化完成！
2025-01-17T00:00:00.000000000中性化完成！
2025-01-20T00:00:00.000000000中性化完成！
2025-01-21T00:00:00.000000000中性化完成！
2025-01-22T00:00:00.000000000中性化完成！
2025-01-23T00:00:00.000000000中性化完成！
2025-01-24T00:00:00.000000000中性化完成！
2025-01-27T00:00:00.000000000中性化完成！
2025-02-05T00:00:00.000000000中性化完成！
2025-02-06T00:00:00.000000000中性化完成！
2025-02-07T00:00:00.000000000中性化完成！
2025-02-10T00:00:00.000000000中性化完成！
2025-02-11T00:00:00.000000000中性化完成！
2025-02-12T00:00:00.000000000中性化完成！
2025-02-13T00:00:00.000000000中性化完成！
2025-02-14T00:00:00.000000000中性化完成！
2025-02-17T00:00:00.000000000中性化完成！
2025-02-18T00:00:00.00000000

2025-12-12T00:00:00.000000000中性化完成！
2025-12-15T00:00:00.000000000中性化完成！
2025-12-16T00:00:00.000000000中性化完成！
2025-12-17T00:00:00.000000000中性化完成！
2025-12-18T00:00:00.000000000中性化完成！
2025-12-19T00:00:00.000000000中性化完成！
2025-12-22T00:00:00.000000000中性化完成！
2025-12-23T00:00:00.000000000中性化完成！
2025-12-24T00:00:00.000000000中性化完成！
2025-12-25T00:00:00.000000000中性化完成！
2025-12-26T00:00:00.000000000中性化完成！
2025-12-29T00:00:00.000000000中性化完成！
2025-12-30T00:00:00.000000000中性化完成！
2025-12-31T00:00:00.000000000中性化完成！


In [16]:
df4=df4.drop(["open"],axis = 1)
df4.head()

,time,code,close,日内收益率,volume,turnover_ratio,pe_ratio,pb_ratio,ps_ratio,pcf_ratio,roe,net_profit_margin,inc_net_profit_year_on_year,net_operate_cash_flow,circulating_market_cap
0,2025-01-02,000001.XSHE,10.85,-0.026032,-2.256970,-2.112075,-1.565787,-0.063302,-0.074557,-1.354803,1.504460,1.638136,0.977693,-0.944657,1.731194
1,2025-01-02,000002.XSHE,7.11,-0.019310,0.531564,0.365432,0.487044,0.949528,1.495057,0.217562,0.802671,0.236440,0.658079,1.556820,-0.693255
2,2025-01-02,000063.XSHE,36.02,-0.060843,-0.785244,-0.811384,-1.022391,-1.139433,-0.702402,-0.887764,0.714947,1.283861,0.264707,-0.049719,0.706712
3,2025-01-02,000100.XSHE,4.90,-0.016064,-0.010651,-0.192007,-0.297862,-0.886226,-0.231518,-0.700948,-0.469321,0.167125,-0.653161,-0.447469,-0.350280
4,2025-01-02,000157.XSHE,6.58,-0.029499,0.918861,0.984809,1.231699,1.012830,0.396327,1.011529,0.057020,-0.918804,0.330269,0.537585,-1.021096


In [17]:
df_list = []
for code in df4["code"].unique():
    split_data = df4[df4["code"]==code].copy()
    split_data = split_data.ffill()
    split_data["1M"] = split_data["close"].pct_change(1).shift(-1)
    split_data["3M"] = split_data["close"].pct_change(3).shift(-3)
    split_data["5M"] = split_data["close"].pct_change(5).shift(-5)
    split_data["10M"] = split_data["close"].pct_change(10).shift(-10)
    df_list.append(split_data)
all_price_data = pd.concat(df_list)
all_price_data = all_price_data.sort_values(by = ['time','code'])

In [18]:
all_price_data.head()

,time,code,close,日内收益率,volume,turnover_ratio,pe_ratio,pb_ratio,ps_ratio,pcf_ratio,roe,net_profit_margin,inc_net_profit_year_on_year,net_operate_cash_flow,circulating_market_cap,1M,3M,5M,10M
0,2025-01-02,000001.XSHE,10.85,-0.026032,-2.256970,-2.112075,-1.565787,-0.063302,-0.074557,-1.354803,1.504460,1.638136,0.977693,-0.944657,1.731194,-0.003687,0.007373,-0.002765,0.012903
1,2025-01-02,000002.XSHE,7.11,-0.019310,0.531564,0.365432,0.487044,0.949528,1.495057,0.217562,0.802671,0.236440,0.658079,1.556820,-0.693255,-0.015471,-0.008439,-0.022504,-0.032349
2,2025-01-02,000063.XSHE,36.02,-0.060843,-0.785244,-0.811384,-1.022391,-1.139433,-0.702402,-0.887764,0.714947,1.283861,0.264707,-0.049719,0.706712,-0.029428,-0.035258,0.000000,0.000000
3,2025-01-02,000100.XSHE,4.90,-0.016064,-0.010651,-0.192007,-0.297862,-0.886226,-0.231518,-0.700948,-0.469321,0.167125,-0.653161,-0.447469,-0.350280,-0.014286,0.024490,-0.010204,0.000000
4,2025-01-02,000157.XSHE,6.58,-0.029499,0.918861,0.984809,1.231699,1.012830,0.396327,1.011529,0.057020,-0.918804,0.330269,0.537585,-1.021096,0.003040,-0.006079,-0.019757,0.048632


In [19]:
all_price_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2430 entries, 0 to 2429
Data columns (total 19 columns):
time                           2430 non-null datetime64[ns]
code                           2430 non-null object
close                          2430 non-null float64
日内收益率                          2430 non-null float64
volume                         2430 non-null float64
turnover_ratio                 2430 non-null float64
pe_ratio                       2430 non-null float64
pb_ratio                       2430 non-null float64
ps_ratio                       2430 non-null float64
pcf_ratio                      2430 non-null float64
roe                            2430 non-null float64
net_profit_margin              2430 non-null float64
inc_net_profit_year_on_year    2430 non-null float64
net_operate_cash_flow          2430 non-null float64
circulating_market_cap         2430 non-null float64
1M                             2420 non-null float64
3M                             2400 non

In [20]:
out_path="第一节代码导出的数据.csv"
all_price_data.to_csv(out_path)